<a href="https://colab.research.google.com/github/aayurchik/27_toxicity_prediction/blob/main/models/phys_chem_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import warnings
warnings.filterwarnings("ignore")
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score)
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

In [4]:
multi_target_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/multi_target_cleaned.csv')
multi_target_df.head(5)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/multi_target_cleaned.csv'

In [ ]:
desc_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/smiles_physchem_descriptors.csv')
desc_df.head(5)

,smiles,BCUT2D_LOGPHI,BCUT2D_LOGPLOW,BalabanJ,BertzCT,Chi0,Chi0n,Chi0v,Chi1,Chi1n,...,VSA_EState1,VSA_EState10,VSA_EState2,VSA_EState3,VSA_EState4,VSA_EState5,VSA_EState6,VSA_EState7,VSA_EState8,VSA_EState9
0,CC(=O)OCC[N+](C)(C)C,1.884911,-2.373581,3.290618,103.696943,8.198671,7.177924,7.177924,4.416502,3.354792,...,5.590000,0.000000,10.300428,0.000000,0.000000,-0.200509,0.000000,0.000000,2.799676,6.177072
1,C[N+](C)(C)CC(=O)[O-],1.844465,-2.445326,3.551253,82.701658,6.784457,5.470817,5.470817,3.416502,2.419670,...,0.418981,0.000000,9.888194,9.888194,0.000000,-1.002315,0.000000,0.000000,0.069444,5.404167
2,O=C(NC(CO)C(O)c1ccc([N+](=O)[O-])cc1)C(Cl)Cl,2.203149,-2.435885,2.821158,447.431584,15.284093,10.070873,11.582731,9.362280,5.482716,...,0.000000,10.701515,19.911925,31.946182,0.169222,-0.756162,4.053667,-1.252928,-0.551198,0.000000
3,O=C(O)c1ccccc1O,2.138185,-1.945909,3.152941,245.970152,7.560478,5.112077,5.112077,4.715214,2.728688,...,0.000000,0.000000,10.261759,17.305741,-0.067130,-1.311944,5.811574,0.000000,0.000000,0.000000
4,CC(NC(C)(C)C)C(=O)c1cccc(Cl)c1,2.130846,-2.455588,2.784739,365.869855,12.344935,10.172964,10.928893,7.293512,5.369174,...,0.000000,5.854074,12.053441,3.835666,0.574026,0.067060,6.831265,0.000000,7.978912,0.000000


In [ ]:
# объединяем по SMILES
merged_df = multi_target_df.merge(desc_df, on='smiles', how='inner')

# Посчитаем абсолютную корреляцию каждого физико-химического признака с каждой категорией токсичности
# Отберём только те признаки, которые хотя бы с одной категорией имеют |r| > 0.1
# Посчитаем, сколько таких признаков оказалось

descriptor_cols = desc_df.columns.drop('smiles')
target_cols = multi_target_df.columns.drop('smiles')

# считаем корреляцию по каждому target на своих валидных строках
corr_dict = {}
for cat in target_cols:
    valid_idx = merged_df[cat].notna()
    corr_dict[cat] = merged_df.loc[valid_idx, descriptor_cols].corrwith(
        merged_df.loc[valid_idx, cat])
corr_df = pd.DataFrame(corr_dict)
threshold = 0.1
informative_features = corr_df[(corr_df.abs() > threshold).any(axis=1)].index.tolist()
print("Информативных признаков:", len(informative_features))
# Убираем сильно коррелированные (r > 0.9)
corr_matrix = merged_df[informative_features].corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > 0.9)]
final_features = [f for f in informative_features if f not in to_drop]
print("Оставшиеся признаки:", len(final_features))
final_df = merged_df[['smiles'] + final_features + list(target_cols)]

Информативных признаков: 99
Оставшиеся признаки: 78


In [ ]:
from sklearn.impute import SimpleImputer
final_df.isna().sum()

# Таргеты:
# 1 = токсична по этому классу
# 0 = не токсична
# NaN = молекулу не тестировали

# Заполняем пропуски в признаках медианой (медиана лучший вариант для химических признаков)
imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(final_df[final_features])
# Таргеты не трогаем, NaN это норма.
y = final_df[target_cols]


**Метрика**  
Почему именно F1:
- F1-macro считает F1 для каждого токс-класса и усредняет. F1-micro суммирует TP/FP/FN по всем классам. В токсикологии обычно берут F1-macro, потому что редкие токс-типы не должны теряться.
- классы очень несбалансированы, accuracy не подходит.  
- важно не пропустить токсичные молекулы, нужен высокий Recall.  
- важно не давать слишком много ложных токсичных, нужен Precision.  

F1 = 2 × (Precision × Recall) / (Precision + Recall)  
Precision = TP / (TP + FP)  
Recall = TP / (TP + FN)  


ключевая метрика = F1-macro  
Доп. метрики:  
- F1-micro  
- ROC-AUC macro  
- Precision / Recall для каждого класса  

In [ ]:
# Масштабирование признаков
# Стандартизация
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
def stratified_split(X, y_target):
    """
    Пропускаем NaN (молекулы, которые не тестировались на данную токсичность).
    Возвращает None если недостаточно данных для обучения (только один класс).
    """
    valid_idx = y_target.notna()
    X_valid = X[valid_idx]
    y_valid = y_target[valid_idx]
    # Проверка: нужны оба класса (0 и 1) для обучения бинарного классификатора
    if len(np.unique(y_valid)) < 2:
        return None
    return train_test_split(
        X_valid, y_valid,
        test_size=0.2,
        random_state=42,
        stratify=y_valid)  # Сохраняем распределение классов

def train_and_evaluate_single_target(X, y_target, model):
    """
    Обучает модель на одной категории токсичности и возвращает метрики.
    """
    split = stratified_split(X, y_target)
    if split is None:
        return None  # Недостаточно данных для обучения
    X_train, X_test, y_train, y_test = split
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    # Ключевые метрики для оценки бинарной классификации
    metrics = {
        "F1": f1_score(y_test, y_pred),        # Основная метрика
        "Precision": precision_score(y_test, y_pred),  # Точность
        "Recall": recall_score(y_test, y_pred),}        # Полнота
    # ROC-AUC показывает качество ранжирования, но не требует выбора порога
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
        metrics["ROC-AUC"] = roc_auc_score(y_test, y_prob)
    else:
        metrics["ROC-AUC"] = None

    return metrics

In [ ]:
# обучение на физхим признаках
results = {}
for cat in target_cols:
    results[cat] = {}
    # Knn
    knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
    results[cat]["KNN"] = train_and_evaluate_single_target(X_scaled, y[cat], knn)
    # Logistic Regression
    lr = LogisticRegression(max_iter=2000, solver="saga", n_jobs=-1)
    results[cat]["LogReg"] = train_and_evaluate_single_target(X_scaled, y[cat], lr)
# Формирование таблицы результатов
table_rows = []
for cat, models in results.items():
    for model_name, metrics in models.items():
        if metrics:  # Добавляем только успешные результаты
            row = {
                "tox_category": cat,
                "model": model_name,
                "F1": metrics["F1"],
                "Precision": metrics["Precision"],
                "Recall": metrics["Recall"],
                "ROC-AUC": metrics["ROC-AUC"]}
            table_rows.append(row)
# Создание и форматирование итоговой таблицы
results_df = pd.DataFrame(table_rows)
results_df = results_df.sort_values(["tox_category", "model"])
# Вывод полной таблицы результатов
pd.set_option('display.max_rows', None)
print(results_df.to_string(index=False))

Категория: acute_toxicity
Категория: carcinogenicity
Категория: cardiotoxicity
Категория: dermal_toxicity
Категория: developmental_toxicity
Категория: endocrine_disruption
Категория: gastrointestinal_toxicity
Категория: genotoxicity
Категория: hematotoxicity
Категория: hepatotoxicity
Категория: immunotoxicity
Категория: metabolic_toxicity
Категория: musculoskeletal_toxicity
Категория: nephrotoxicity
Категория: neurotoxicity
Категория: ocular_toxicity
Категория: other_toxicity
Категория: oxidative_stress
Категория: reproductive_toxicity
Категория: respiratory_toxicity
Категория: sensory_toxicity
РЕЗУЛЬТАТЫ (физико-химические признаки)
          tox_category  model       F1  Precision   Recall  ROC-AUC
        acute_toxicity    KNN 0.557692   0.632153 0.498925 0.791316
        acute_toxicity LogReg 0.520833   0.660066 0.430108 0.787133
       carcinogenicity    KNN 0.702079   0.675556 0.730769 0.673377
       carcinogenicity LogReg 0.753304   0.695122 0.822115 0.712882
        cardiotoxi

- NaN вызваны отсутствием второго класса после разбиения, слишком мало положительных примеров. возможно стоит объединить редкие категории токсичности в более крупную группу (тут в таблице осталось 14 категорий из 20 существующих)

- Низкое качество у cardiotoxicity, endocrine disruption, oxidative stress. Можно добавить фингерпринты или попробовать более мощные модели.

- KNN почти всегда работает лучше логистической регрессии, возможна нелинейная природа.

In [ ]:
# Посчитаем количество 0, 1 и NaN по каждой категории
# посмотрим, какие категории вообще есть, сколько по ним данных, и как их лучше объединять, чтобы избежать NaN в моделях.
summary = []

for cat in target_cols:
    col = multi_target_df[cat]
    n0 = (col == 0).sum()
    n1 = (col == 1).sum()
    nn = col.isna().sum()
    summary.append([cat, n0, n1, nn])

import pandas as pd
summary_df = pd.DataFrame(summary, columns=["category", "num_0", "num_1", "num_nan"])
summary_df.sort_values("num_1")  # сортировка от самых редких


,category,num_0,num_1,num_nan
4,developmental_toxicity,24,164,338867
20,sensory_toxicity,0,643,338412
18,reproductive_toxicity,17,860,338178
8,hematotoxicity,0,872,338183
13,nephrotoxicity,0,894,338161
12,musculoskeletal_toxicity,0,972,338083
11,metabolic_toxicity,0,973,338082
1,carcinogenicity,680,1037,337338
17,oxidative_stress,5486,1134,332435
16,other_toxicity,0,1233,337822


In [ ]:
"""
1. Объединить семантически близкие категории
2. Удалить категории, которые нельзя объединить
3. Проверить, что все итоговые категории имеют оба класса (0 и 1)
"""
# Группы для объединения
merge_groups = {
    # нарушения нервной системы и органов чувств
    'neuro_sensory_toxicity': ['neurotoxicity', 'sensory_toxicity'],
    # поражение иммунной системы и крови
    'immuno_hematotoxicity': ['immunotoxicity', 'hematotoxicity'],
    # влияние на репродукцию и развитие
    'reprod_dev_toxicity': ['reproductive_toxicity', 'developmental_toxicity'],
    # нарушения гормональной и метаболической систем
    'endocrine_metabolic_tox': ['endocrine_disruption', 'metabolic_toxicity']}

# Категории для удаления (нет по смыслу близких категорий для объединения)
delete_categories = [
    'musculoskeletal_toxicity',
    'nephrotoxicity',
    'gastrointestinal_toxicity',
    'other_toxicity']

def merge_toxicity_categories(df, merge_mapping, delete_list):
    """
    Объединяет категории токсичности по заданным группам.
    - Если молекула токсична хотя бы в одной из объединяемых категорий = 1
    - Если молекула не токсична во всех объединяемых категориях = 0
    - Если по всем категориям нет данных = NaN
    """
    df_merged = df.copy()
    for new_name, old_names in merge_mapping.items():
        # Создаём новую колонку для объединённой категории
        df_merged[new_name] = np.nan
        # Создаём маски для всех старых категорий
        any_one_mask = pd.Series(False, index=df_merged.index)
        any_zero_mask = pd.Series(False, index=df_merged.index)
        all_nan_mask = pd.Series(True, index=df_merged.index)
        for old_name in old_names:
            if old_name in df_merged.columns:
                # Молекула токсична хотя бы в одной категории
                any_one_mask = any_one_mask | (df_merged[old_name] == 1)
                # Молекула не токсична хотя бы в одной категории
                any_zero_mask = any_zero_mask | (df_merged[old_name] == 0)
                # Учитываем NaN для определения полного отсутствия данных
                all_nan_mask = all_nan_mask & df_merged[old_name].isna()
        # Если есть хотя бы одна 1 = 1
        df_merged.loc[any_one_mask, new_name] = 1
        # Если нет 1, но есть хотя бы один 0 = 0
        df_merged.loc[~any_one_mask & any_zero_mask, new_name] = 0
        # Удаляем старые категории
        df_merged = df_merged.drop(columns=[c for c in old_names if c in df_merged.columns])
    existing_to_delete = [cat for cat in delete_list if cat in df_merged.columns]
    if existing_to_delete:
        df_merged = df_merged.drop(columns=existing_to_delete)
    # Категории для обучения все колонки кроме признаков и smiles
    all_cols = df_merged.columns.tolist()
    target_cols_final = [
        col for col in all_cols
        if col not in final_features and col != 'smiles']
    categories_without_zeros = [
        cat for cat in target_cols_final
        if (df_merged[cat] == 0).sum() == 0]

    if categories_without_zeros:
        df_merged = df_merged.drop(columns=categories_without_zeros)
        # Обновляем список категорий
        target_cols_final = [
            col for col in target_cols_final
            if col not in categories_without_zeros]
    return df_merged, target_cols_final

# Применяем функцию объединения
df_merged, target_cols_final = merge_toxicity_categories(
    final_df, merge_groups, delete_categories)
# Признаки остаются теми же (уже масштабированные)
X_final = X_scaled
# Таргеты новые объединённые категории
y_final = df_merged[target_cols_final]

print(f"Итоговое количество категорий: {len(target_cols_final)}")
print(f"Признаков: {X_final.shape[1]}")
print(f"Молекул в датасете: {X_final.shape[0]}")
print()
# Выводим распределение по категориям для проверки
for cat in sorted(target_cols_final):
    n0 = (df_merged[cat] == 0).sum()
    n1 = (df_merged[cat] == 1).sum()
    nn = df_merged[cat].isna().sum()
    total = n0 + n1
    balance = n1 / total if total > 0 else 0
    print(f"{cat:30} | токсичных: {n1:5} | нетоксичных: {n0:5} | "
          f"нет данных: {nn:7} | баланс: {balance:5.1%}")


Итоговое количество категорий: 13
Признаков: 78
Молекул в датасете: 339055

acute_toxicity                 | токсичных:  2323 | нетоксичных:  5350 | нет данных:  331382 | баланс: 30.3%
carcinogenicity                | токсичных:  1037 | нетоксичных:   680 | нет данных:  337338 | баланс: 60.4%
cardiotoxicity                 | токсичных: 22146 | нетоксичных: 298840 | нет данных:   18069 | баланс:  6.9%
dermal_toxicity                | токсичных:  2070 | нетоксичных:   905 | нет данных:  336080 | баланс: 69.6%
endocrine_metabolic_tox        | токсичных:  2448 | нетоксичных:  5487 | нет данных:  331120 | баланс: 30.9%
genotoxicity                   | токсичных:  4850 | нетоксичных:  8244 | нет данных:  325961 | баланс: 37.0%
hepatotoxicity                 | токсичных:  1883 | нетоксичных:  1345 | нет данных:  335827 | баланс: 58.3%
immuno_hematotoxicity          | токсичных:  2549 | нетоксичных:  4933 | нет данных:  331573 | баланс: 34.1%
neuro_sensory_toxicity         | токсичных:  1669 |

In [ ]:
# Обновляем y для использования объединённых категорий
y_final = df_merged[target_cols_final]
# Функции из кода выше
def stratified_split(X, y_target):
    valid_idx = y_target.notna()
    X_valid = X[valid_idx]
    y_valid = y_target[valid_idx]
    classes = np.unique(y_valid)
    if len(classes) < 2:
        return None
    return train_test_split(
        X_valid, y_valid,
        test_size=0.2,
        random_state=42,
        stratify=y_valid)
def train_and_evaluate_single_target(X, y_target, model):
    split = stratified_split(X, y_target)
    if split is None:
        return {
            "F1": None,
            "Precision": None,
            "Recall": None,
            "ROC-AUC": None}
    X_tr, X_te, y_tr, y_te = split
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    metrics = {
        "F1": f1_score(y_te, pred),
        "Precision": precision_score(y_te, pred),
        "Recall": recall_score(y_te, pred),}
    if hasattr(model, "predict_proba"):
        prob = model.predict_proba(X_te)[:, 1]
        metrics["ROC-AUC"] = roc_auc_score(y_te, prob)
    else:
        metrics["ROC-AUC"] = None
    return metrics

# Обучаем модели
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
results = {}
for cat in target_cols_final:
    results[cat] = {}
    # KNN
    knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
    results[cat]["KNN"] = train_and_evaluate_single_target(
        X_scaled, y_final[cat], knn)
    # Logistic Regression
    lr = LogisticRegression(max_iter=2000, solver="saga", n_jobs=-1)
    results[cat]["LogReg"] = train_and_evaluate_single_target(
        X_scaled, y_final[cat], lr)


Модель для acute_toxicity

Модель для carcinogenicity

Модель для cardiotoxicity

Модель для dermal_toxicity

Модель для genotoxicity

Модель для hepatotoxicity

Модель для ocular_toxicity

Модель для oxidative_stress

Модель для respiratory_toxicity

Модель для neuro_sensory_toxicity

Модель для immuno_hematotoxicity

Модель для reprod_dev_toxicity

Модель для endocrine_metabolic_tox
              tox_category   model        F1  Precision    Recall   ROC-AUC
0           acute_toxicity     KNN  0.557692   0.632153  0.498925  0.791316
1           acute_toxicity  LogReg  0.520833   0.660066  0.430108  0.787133
2          carcinogenicity     KNN  0.702079   0.675556  0.730769  0.673377
3          carcinogenicity  LogReg  0.753304   0.695122  0.822115  0.712882
4           cardiotoxicity     KNN  0.516295   0.703545  0.407767  0.845697
5           cardiotoxicity  LogReg  0.218767   0.576108  0.135019  0.840991
6          dermal_toxicity     KNN  0.821002   0.811321  0.830918  0.762925
7  

In [ ]:
# Выше старые результаты, создаем и выводим таблицу  с новыми результатами
table = []
for cat, models in results.items():
    for model_name, metrics in models.items():
        if metrics:  # Добавляем только успешные результаты
            row = {"tox_category": cat, "model": model_name}
            row.update(metrics)
            table.append(row)
new_results_df = pd.DataFrame(table)
new_results_df = new_results_df.sort_values(["tox_category", "model"])
print(new_results_df.to_string(index=False))
# Сохраняем для дальнейшего использования
results_df_merged = new_results_df.copy()

           tox_category  model       F1  Precision   Recall  ROC-AUC
         acute_toxicity    KNN 0.557692   0.632153 0.498925 0.791316
         acute_toxicity LogReg 0.520833   0.660066 0.430108 0.787135
        carcinogenicity    KNN 0.702079   0.675556 0.730769 0.673377
        carcinogenicity LogReg 0.753304   0.695122 0.822115 0.712882
         cardiotoxicity    KNN 0.516295   0.703545 0.407767 0.845697
         cardiotoxicity LogReg 0.218767   0.576108 0.135019 0.840991
        dermal_toxicity    KNN 0.821002   0.811321 0.830918 0.762925
        dermal_toxicity LogReg 0.835189   0.774793 0.905797 0.753556
endocrine_metabolic_tox    KNN 0.481663   0.600610 0.402041 0.720809
endocrine_metabolic_tox LogReg 0.437500   0.654472 0.328571 0.726642
           genotoxicity    KNN 0.759788   0.780435 0.740206 0.880292
           genotoxicity LogReg 0.648713   0.773181 0.558763 0.833884
         hepatotoxicity    KNN 0.766879   0.737745 0.798408 0.756244
         hepatotoxicity LogReg 0.7

Категории которые удалили (были NaN):
- gastrointestinal_toxicity
- musculoskeletal_toxicity
- nephrotoxicity
- other_toxicity

Результаты до и после корректировок:

| Категория | Модель | F1 (ДО) | F1 (ПОСЛЕ) | ΔF1 | Комментарий |
|-----------|--------|---------|------------|-----|-------------|
| **neuro_sensory_toxicity** (neuro + sensory) | | | | | |
| | KNN | 0.9125 | 0.9157 | **+0.0032** | Небольшое улучшение |
| | LogReg | 0.9130 | 0.9119 | -0.0011 | Практически без изменений |
| **immuno_hematotoxicity** (immuno + hema) | | | | | |
| | KNN | 0.5425 | 0.5593 | **+0.0168** | Небольшое улучшение |
| | LogReg | 0.5350 | 0.5279 | -0.0071 | Незначительное ухудшение |
| **reprod_dev_toxicity** (reprod + dev) | | | | | |
| | KNN | 0.9914 | 0.9808 | -0.0106 | Небольшое снижение |
| | LogReg | 0.9914 | 0.9783 | -0.0131 | Небольшое снижение |
| **endocrine_metabolic_tox** (endo + metabolic) | | | | | |
| | KNN | 0.4897 | 0.4817 | -0.0080 | Небольшое снижение |
| | LogReg | 0.3393 | 0.4375 | **+0.0982** | Хорошее улучшение |

Объединение категорий имело смысл, двигаемся дальше, подключая фингерпринты.

In [ ]:
cat = 'neuro_sensory_toxicity'
model = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)

X = X_scaled
y_target = y_final[cat]

model.fit(X_valid, y_valid)


NameError: name 'KNeighborsClassifier' is not defined